# Klasifikasi Sampah Organik & Anorganik Menggunakan Convolutional Neural Network (CNN)

Tugas ini bertujuan untuk membangun model kecerdasan buatan berbasis **Deep Learning** yang mampu mengenali dan mengklasifikasikan gambar sampah menjadi dua kategori:
1. **Organic (O)** - Sampah Organik (sisa makanan, daun, kertas, dll.)
2. **Recyclable (R)** - Sampah Anorganik/Daur Ulang (plastik, botol, logam, dll.)

### Tahapan Pengerjaan:
1. **Inisialisasi & Import Library**
2. **Konfigurasi Parameter**
3. **Preprocessing & Augmentasi Data** (Mendukung unduh otomatis di Google Colab)
4. **Desain Arsitektur CNN**
5. **Pelatihan Model**
6. **Visualisasi Riwayat Pelatihan (Accuracy & Loss)**
7. **Evaluasi Model pada Test Set (Confusion Matrix & Classification Report)**
8. **Prediksi Gambar Baru (Inference)**
9. **Demo Web Frontend Interaktif (Gradio GUI)**

### Sel 1: Inisialisasi & Import Library
Kita mengimpor pustaka utama yang digunakan seperti TensorFlow/Keras untuk deep learning, NumPy untuk komputasi numerik, serta Matplotlib dan Seaborn untuk visualisasi data.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix

# Set seed untuk reproduksibilitas
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Tersedia:", tf.config.list_physical_devices('GPU'))

### Sel 2: Konfigurasi Parameter
Definisikan parameter pelatihan seperti ukuran gambar (`IMAGE_SIZE`), jumlah data per batch (`BATCH_SIZE`), jumlah epoch (`EPOCHS`), serta jalur direktori dataset.

In [ ]:
# Parameter Konfigurasi
IMAGE_SIZE = (150, 150)  # Gambar akan di-resize menjadi 150x150
BATCH_SIZE = 32          # Jumlah sampel per iterasi
EPOCHS = 15              # Batas maksimal epoch pelatihan
DATASET_DIR = "dataset/DATASET"  # Folder dataset utama (default lokal)
MODEL_SAVE_PATH = "model_waste_cnn.keras"  # Lokasi penyimpanan model terbaik

### Sel 3: Preprocessing & Augmentasi Data
Bagian ini memuat gambar secara otomatis. Jika Anda menjalankan notebook ini di **Google Colab**, sel ini secara pintar akan mendeteksi lingkungan awan tersebut dan **mengunduh dataset secara otomatis** dari server Kaggle (menggunakan `kagglehub`) tanpa perlu mengunggah file gambar secara manual. Jika dijalankan secara **lokal di komputer**, program akan langsung menggunakan folder lokal `dataset/` yang sudah Anda miliki.

In [ ]:
# Deteksi apakah berjalan di Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[INFO] Berjalan di Google Colab. Mengunduh dataset otomatis...")
    # Colab biasanya sudah memiliki kagglehub, jika tidak kita install
    try:
        import kagglehub
    except ImportError:
        os.system("pip install kagglehub")
        import kagglehub
    
    # Unduh dataset dari Kaggle (publik, tidak butuh kredensial)
    downloaded_path = kagglehub.dataset_download("techsash/waste-classification-data")
    DATASET_DIR = os.path.join(downloaded_path, 'DATASET')
    print("[SUKSES] Dataset berhasil diunduh di Google Colab pada:", DATASET_DIR)

train_dir = os.path.join(DATASET_DIR, 'TRAIN')
test_dir = os.path.join(DATASET_DIR, 'TEST')

# Penanganan variasi folder bersarang (nested) di Kaggle
if not os.path.exists(train_dir) or not os.path.exists(test_dir):
    alt_train_dir = os.path.join(DATASET_DIR, 'DATASET', 'TRAIN')
    alt_test_dir = os.path.join(DATASET_DIR, 'DATASET', 'TEST')
    if os.path.exists(alt_train_dir) and os.path.exists(alt_test_dir):
        train_dir = alt_train_dir
        test_dir = alt_test_dir

# Validasi akhir
if not os.path.exists(train_dir) or not os.path.exists(test_dir):
    print(f"[EROR] Folder dataset tidak ditemukan di '{DATASET_DIR}'.")
    print("Jika dijalankan secara lokal, pastikan Anda telah mengekstrak dataset di folder 'dataset/DATASET'")
    train_ds = None
else:
    # Load Training Set (di-split 80% training, 20% validation untuk memantau overfitting)
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE
    )

    # Load Test Set untuk evaluasi akhir
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False  # Jangan di-shuffle untuk memudahkan pencocokan label evaluasi
    )

    class_names = train_ds.class_names
    print("\nKelas yang terdeteksi:", class_names)  # ['O', 'R'] -> O: Organic, R: Recyclable

    # Mendefinisikan teknik Augmentasi Data Gambar
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
    ])
    print("Pipeline preprocessing dan augmentasi berhasil disiapkan.")

    # Optimasi performa I/O data menggunakan Prefetch
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

### Sel 4: Desain Arsitektur CNN
Kita merancang model Convolutional Neural Network (CNN) sekuensial:
1. **Rescaling:** Mengubah nilai piksel dari rentang [0, 255] menjadi [0, 1].
2. **Layer Konvolusi (`Conv2D` & `MaxPooling2D`):** Mengekstrak fitur-fitur penting seperti garis, bentuk, tekstur, dan menyusutkan ukuran dimensi spasial.
3. **Flatten & Dense Layers:** Mengubah matriks fitur menjadi vektor satu dimensi dan melakukan pembelajaran klasifikasi.
4. **Dropout:** Mematikan neuron secara acak sebesar 50% untuk mencegah model terlalu menghafal data latih (*overfitting*).
5. **Output Layer:** Satu neuron dengan fungsi aktivasi `sigmoid` (menghasilkan nilai probabilitas antara 0 dan 1).

In [ ]:
def build_model():
    model = models.Sequential([
        # Layer Rescaling untuk normalisasi input gambar ke [0, 1]
        layers.Rescaling(1./255, input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
        
        # Blok Konvolusi 1
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Blok Konvolusi 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Blok Konvolusi 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Blok Konvolusi 4
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Flatten & Fully Connected Classifier
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),  # Regularisasi anti-overfitting
        
        # Output biner: < 0.5 -> Organik (O), >= 0.5 -> Anorganik/Daur Ulang (R)
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compile model menggunakan optimizer Adam dan loss Binary Crossentropy
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

### Sel 5: Pelatihan Model
Latih model dengan data latih dan pantau menggunakan data validasi. Kita menggunakan callback `EarlyStopping` agar jika performa model di data validasi sudah tidak membaik selama beberapa epoch (patience=5), proses pelatihan dihentikan secara otomatis untuk menghemat waktu dan mencegah overfitting. Kita juga menggunakan `ModelCheckpoint` untuk menyimpan model dengan performa terbaik.

In [ ]:
# Konfigurasi Callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor='val_loss',
    save_best_only=True
)

# Memulai proses training
if 'train_ds' in locals() and train_ds is not None:
    print("Memulai pelatihan model...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[early_stopping, checkpoint]
    )
    print("Pelatihan selesai! Model terbaik disimpan di:", MODEL_SAVE_PATH)
else:
    print("[INFO] Sel latih dilewati karena folder dataset belum siap di direktori lokal.")

### Sel 6: Visualisasi Riwayat Pelatihan
Buat plot grafik loss dan akurasi antara data latih (training) dan data validasi untuk menganalisis perkembangan model.

In [ ]:
if 'history' in locals() and history is not None:
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 5))
    
    # Plot Akurasi
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', color='#2ca02c', linewidth=2)
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#d62728', linewidth=2)
    plt.legend(loc='lower right')
    plt.title('Grafik Akurasi Model')
    plt.xlabel('Epoch')
    plt.ylabel('Akurasi')
    plt.grid(True, linestyle='--', alpha=0.5)

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', color='#2ca02c', linewidth=2)
    plt.plot(epochs_range, val_loss, label='Validation Loss', color='#d62728', linewidth=2)
    plt.legend(loc='upper right')
    plt.title('Grafik Loss Model')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300)
    plt.show()
else:
    print("[INFO] Riwayat pelatihan tidak tersedia karena model tidak dilatih di sesi ini.")

### Sel 7: Evaluasi Model & Confusion Matrix
Uji kinerja model terbaik menggunakan data pengujian (Test Set) yang sama sekali baru. Kita akan menampilkan akurasi test set, classification report (Precision, Recall, F1-Score), serta Confusion Matrix.

In [ ]:
if 'test_ds' in locals() and test_ds is not None and os.path.exists(MODEL_SAVE_PATH):
    # Memuat model terbaik yang telah disimpan
    saved_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
    
    # Evaluasi pada data uji
    test_loss, test_acc = saved_model.evaluate(test_ds)
    print(f"\nAkurasi pada Test Set: {test_acc * 100:.2f}%")
    print(f"Loss pada Test Set: {test_loss:.4f}")

    # Memprediksi seluruh data pengujian
    y_true = []
    y_pred = []

    # Loop untuk mengambil data batch demi batch
    for images, labels in test_ds:
        preds = saved_model.predict(images, verbose=0)
        preds_binary = (preds >= 0.5).astype(int).flatten()
        y_true.extend(labels.numpy())
        y_pred.extend(preds_binary)

    # Membuat Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=['Organic (O)', 'Recyclable (R)'], 
                yticklabels=['Organic (O)', 'Recyclable (R)'])
    plt.ylabel('Aktual (Label Asli)')
    plt.xlabel('Prediksi Model')
    plt.title('Confusion Matrix Evaluasi Sampah')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300)
    plt.show()

    # Menampilkan Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Organic (O)', 'Recyclable (R)']))
else:
    print("[INFO] Model atau data pengujian tidak ditemukan di sesi ini.")

### Sel 8: Prediksi Gambar Baru (Inference)
Buat fungsi untuk menguji model dengan gambar acak baru. Anda hanya perlu memberikan path ke berkas gambar Anda, lalu fungsi ini akan memprosesnya dan menampilkan hasil klasifikasinya.

In [ ]:
def predict_new_image(image_path):
    if not os.path.exists(image_path):
        print(f"[ERROR] File gambar tidak ditemukan di '{image_path}'")
        return
    
    # Load gambar & resize
    img = image.load_img(image_path, target_size=IMAGE_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Ubah bentuk jadi (1, 150, 150, 3)

    if os.path.exists(MODEL_SAVE_PATH):
        saved_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
        prediction = saved_model.predict(img_array, verbose=0)[0][0]
        
        # Klasifikasi berdasarkan threshold 0.5
        if prediction >= 0.5:
            label = "Recyclable (Anorganik/Daur Ulang)"
            confidence = prediction * 100
        else:
            label = "Organic (Organik)"
            confidence = (1 - prediction) * 100
            
        # Tampilkan visualisasi gambar dan hasil prediksi
        plt.figure(figsize=(5, 5))
        plt.imshow(img)
        plt.title(f"Prediksi: {label}\nKeyakinan: {confidence:.2f}%")
        plt.axis('off')
        plt.show()
        
        print(f"[HASIL] Gambar diidentifikasi sebagai '{label}' dengan keyakinan {confidence:.2f}%")
    else:
        print("[ERROR] Model terbaik tidak ditemukan. Harap latih model terlebih dahulu.")

# --- Petunjuk Penggunaan ---
# Untuk melakukan pengujian gambar baru, letakkan file gambar di folder proyek,
# kemudian uncomment baris di bawah ini dan isi dengan path gambar Anda:
# predict_new_image("contoh_sampah.jpg")

### Sel 9: Demo Web Frontend Interaktif (Gradio GUI)
Gunakan sel ini untuk meluncurkan antarmuka web interaktif (GUI). Anda dapat mengunggah gambar langsung dari komputer atau HP, dan melihat hasil prediksi model secara real-time. Jika dijalankan di Google Colab, Gradio akan secara otomatis memberikan tautan publik (*Public URL*) berkekuatan HTTPS (contoh: `xxxx.gradio.live`) sehingga Anda dapat mencobanya langsung di HP atau membagikannya ke dosen penguji Anda!

In [ ]:
# Colab membutuhkan instalasi gradio jika belum ada
if IN_COLAB:
    import os
    os.system("pip install gradio")

import gradio as gr
from PIL import Image

def classify_waste_gui(input_image):
    if not os.path.exists(MODEL_SAVE_PATH):
        return {"Error": 1.0}, "Berkas model tidak ditemukan. Silakan jalankan proses training di Sel 5 terlebih dahulu."
    
    # Load model terbaik
    model = tf.keras.models.load_model(MODEL_SAVE_PATH)
    
    # Preprocessing
    img = input_image.resize(IMAGE_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # Prediksi
    prediction = model.predict(img_array, verbose=0)[0][0]
    
    prob_recyclable = float(prediction)
    prob_organic = float(1 - prediction)
    
    if prediction >= 0.5:
        decision = f"Hasil: ANORGANIK / DAUR ULANG (Recyclable) - Keyakinan: {prob_recyclable * 100:.2f}%"
    else:
        decision = f"Hasil: ORGANIK - Keyakinan: {prob_organic * 100:.2f}%"
        
    outputs = {
        "Organic (Organik)": prob_organic,
        "Recyclable (Anorganik/Daur Ulang)": prob_recyclable
    }
    return outputs, decision

# Jalankan web app
theme = gr.themes.Soft(primary_hue="green", secondary_hue="emerald")
app_gui = gr.Interface(
    fn=classify_waste_gui,
    inputs=gr.Image(type="pil", label="Unggah Gambar Sampah"),
    outputs=[
        gr.Label(num_top_classes=2, label="Skor Probabilitas"),
        gr.Textbox(label="Kesimpulan Model")
    ],
    title="🟢 Waste Classifier GUI (CNN Deep Learning)",
    description="Antarmuka web interaktif klasifikasi sampah organik vs anorganik.",
    theme=theme
)

# share=True akan menghasilkan link publik gratis yang bisa dibuka di perangkat lain selama runtime aktif!
app_gui.launch(share=True, inline=True)